## Exercise 1: Ingest Raw Booking Data (Bronze Layer)

The **Bronze layer** stores raw, unmodified data exactly as it arrives from source systems. In a production pipeline, data would arrive from a Property Management System (PMS). For this lab, a sample dataset representing bookings across five GlobStay properties is provided for you.

Data issues are intentional — they reflect real-world conditions you will address in the Silver layer.

### ✅ Provided: Create the Unity Catalog structure and sample data

Run the two cells below to:
- Create a catalog `hospitality_lab` with `bronze`, `silver`, and `gold` schemas
- Create a sample raw bookings DataFrame containing deliberate quality issues

Inspect the output carefully — you will need to understand the data issues before tackling Exercise 2.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ Run this cell — creates the Unity Catalog structure

catalog = "hospitality_lab"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{storage_root}' COMMENT 'GlobStay hospitality analytics platform'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze COMMENT 'Raw ingested booking data'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver COMMENT 'Cleaned and validated booking data'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold   COMMENT 'Analytics-ready aggregated data'")

print(f"✅ Catalog '{catalog}' and schemas bronze / silver / gold created.")

In [ ]:
# ✅ Run this cell — creates the raw bookings DataFrame

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("booking_id",      StringType(),  True),
    StructField("property_id",     StringType(),  True),
    StructField("property_name",   StringType(),  True),
    StructField("check_in_date",   StringType(),  True),
    StructField("check_out_date",  StringType(),  True),
    StructField("room_type",       StringType(),  True),
    StructField("rate_per_night",  DoubleType(),  True),
    StructField("guest_id",        StringType(),  True),
    StructField("guest_name",      StringType(),  True),
    StructField("guest_email",     StringType(),  True),
    StructField("booking_status",  StringType(),  True),
    StructField("channel",         StringType(),  True),
    StructField("adults",          IntegerType(), True),
    StructField("children",        IntegerType(), True),
])

raw_data = [
    # (booking_id, property_id, property_name, check_in, check_out, room_type, rate, guest_id, guest_name, guest_email, status, channel, adults, children)
    ("B001","P01","Grand Harbor Hotel",   "2025-06-15","2025-06-18","Deluxe",  250.00,"G001","Alex Rivera",      "alex.r@email.com",    "confirmed","direct",    2,0),
    ("B002","P01","Grand Harbor Hotel",   "2025-06-20","2025-06-22","Suite",   450.00,"G002","Maria Santos",     "m.santos@corp.com",   "confirmed","corporate", 1,0),
    ("B003","P02","Sunset Beach Resort",  "2025-07-01","2025-07-08","Sea View",180.00,"G003","James Chen",       "jchen@email.com",     "confirmed","ota",       2,2),
    ("B004","P02","Sunset Beach Resort",  "2025-07-10","2025-07-12","Garden",  140.00,"G004","Sarah Johnson",    None,                  "confirmed","direct",    2,0),  # null email
    ("B005","P03","Mountain Pines Lodge", "2025-08-05","2025-08-10",None,       120.00,"G005","Robert Kim",       "rkim@email.com",      "cancelled","ota",       1,0),  # null room_type
    ("B006","P03","Mountain Pines Lodge", "2025-08-15","2025-08-12","Standard",110.00,"G006","Emily Brown",      "ebrown@email.com",    "confirmed","direct",    2,1),  # check_out < check_in
    ("B001","P01","Grand Harbor Hotel",   "2025-06-15","2025-06-18","Deluxe",  250.00,"G001","Alex Rivera",      "alex.r@email.com",    "confirmed","direct",    2,0),  # DUPLICATE of B001
    ("B007","P04","City Center Inn",      "2025-09-10","2025-09-11","Standard",-50.00,"G007","Michael Turner",   "mturner@email.com",   "confirmed","corporate", 1,0),  # negative rate
    ("B008","P04","City Center Inn",      "2025-09-15","2025-09-18","Superior",195.00,"G008","Lisa Wong",        "lwong@email.com",     "confirmed","ota",       2,0),
    ("B009","P01","Grand Harbor Hotel",   "2025-10-01","2025-10-05","Suite",   480.00,"G009","Carlos Mendez",    "cmendes@biz.com",     "confirmed","corporate", 2,0),
    ("B010","P02","Sunset Beach Resort",  "2025-11-20","2025-11-25","Sea View",155.00,"G010","Aisha Patel",      "aisha.p@email.com",   "confirmed","direct",    2,1),
    ("B011","P03","Mountain Pines Lodge", "2025-12-22","2025-12-26","Deluxe",  180.00,"G011","Thomas Muller",    "tmuller@email.com",   "confirmed","ota",       2,0),
    ("B012","P04","City Center Inn",      "2025-10-08","2025-10-09","Standard",  0.00,"G012","Yuki Tanaka",      "ytanaka@email.com",   "confirmed","direct",    1,0),  # zero rate
    ("B013","P01","Grand Harbor Hotel",   "2025-07-04","2025-07-07","Deluxe",  275.00,"G013","Sophie Anderson",  "sanderson@travel.com","confirmed","ota",       2,2),
    ("B014","P02","Sunset Beach Resort",  "2025-08-20","2025-08-27","Suite",   350.00,"G014","Daniel Hart",      "dhart@email.com",     "pending",  "direct",    2,0),
    ("B015","P05","Airport Transit Hotel","2025-06-30","2025-07-01","Standard", 99.00,"G015","Priya Sharma",     None,                  "confirmed","ota",       1,0),  # null email
]

raw_df = spark.createDataFrame(raw_data, schema)
raw_df.display()
print(f"\nRaw records: {raw_df.count()}")

### 🎯 Challenge: Write raw data to a Bronze table

Write `raw_df` to a Delta table named **`bookings_bronze`** in the `hospitality_lab.bronze` schema.

**Requirements:**
- Use `overwrite` mode so the cell is re-runnable
- After writing, verify the record count matches the raw DataFrame

> 🤖 **Databricks Assistant tip:** Ask *"How do I write a PySpark DataFrame to a Unity Catalog table using saveAsTable with overwrite mode?"*

In [ ]:
# YOUR CODE HERE
# Write raw_df to hospitality_lab.bronze.bookings_bronze (overwrite mode)
# Then verify the record count


## Exercise 2: Clean and Validate Booking Data (Silver Layer)

The **Silver layer** contains cleaned, validated data that downstream processes can rely on. Each quality issue present in the bronze layer must be addressed:

| Issue | Found in | Rule |
|---|---|---|
| Duplicate bookings | B001 appears twice | Keep only the first occurrence per `booking_id` |
| Null `room_type` | B005 | Exclude records with no room type |
| Invalid date order | B006 (`check_out` before `check_in`) | Exclude records where `check_out_date <= check_in_date` |
| Non-positive rate | B007 (–50), B012 (0) | Exclude records where `rate_per_night <= 0` |

> **Note:** Null `guest_email` values (B004, B015) are acceptable — not all bookings require an email address.

### 🎯 Challenge: Create the Silver table

Read from `hospitality_lab.bronze.bookings_bronze`, apply all four cleaning rules, and write the result to **`hospitality_lab.silver.bookings_silver`** (overwrite mode).

**Hints:**
- Use `.dropDuplicates(["booking_id"])` to deduplicate
- Use `.filter(col("room_type").isNotNull())` to remove nulls
- Import `to_date` and `datediff` from `pyspark.sql.functions` to compare dates
- Chain `.filter()` calls for each rule

After writing, print both the Bronze and Silver record counts to confirm records were removed.

> 🤖 **Databricks Assistant tip:** Ask *"How do I filter PySpark rows where check_out_date is not after check_in_date using date comparison?"*

In [ ]:
# YOUR CODE HERE
# Read from bronze, apply cleaning rules, write to silver
# Print bronze count vs silver count


## Exercise 3: Build Analytics-Ready Data (Gold Layer)

The **Gold layer** contains highly curated, business-facing datasets. GlobStay leadership wants two analytics tables:

1. **`property_performance`** — total confirmed bookings, nights sold, and revenue per hotel property
2. **`channel_performance`** — booking volume and revenue broken down by booking channel (direct, OTA, corporate)


### 🎯 Challenge 3a: Property performance table

Create a Gold table `hospitality_lab.gold.property_performance` using Spark SQL.

**Requirements:**
- Source: `hospitality_lab.silver.bookings_silver`
- Filter to `booking_status = 'confirmed'` only
- Columns: `property_id`, `property_name`, `confirmed_bookings`, `total_nights`, `total_revenue`
- Calculate `total_nights` using `DATEDIFF(check_out_date, check_in_date)`
- Calculate `total_revenue` as `total_nights × rate_per_night`
- Order by `total_revenue DESC`

> 🤖 **Databricks Assistant tip:** Ask *"Write a Spark SQL CREATE OR REPLACE TABLE statement that aggregates booking revenue per property using DATEDIFF and SUM."*

In [ ]:
# YOUR CODE HERE — use %sql or spark.sql()
# Create hospitality_lab.gold.property_performance


### 🎯 Challenge 3b: Booking channel performance table

Create a Gold table `hospitality_lab.gold.channel_performance` showing how each booking channel contributes to GlobStay revenue.

**Requirements:**
- Source: `hospitality_lab.silver.bookings_silver`
- Filter to `booking_status = 'confirmed'` only
- Columns: `channel`, `booking_count`, `total_revenue`, `avg_revenue_per_booking` (rounded to 2 decimal places)
- Order by `total_revenue DESC`

> 🤖 **Databricks Assistant tip:** Ask *"How do I calculate average revenue per booking in Spark SQL using ROUND and AVG?"*

In [ ]:
# YOUR CODE HERE
# Create hospitality_lab.gold.channel_performance


## Exercise 4: Error Handling in Pipeline Notebooks

When a notebook runs as a Lakeflow Job task, it must handle errors gracefully and communicate its outcome to downstream tasks. Two key patterns are:

1. **`try/except` with `PySparkException`** — catch specific Spark errors and handle them without crashing the notebook
2. **`dbutils.notebook.exit()`** — signal success or failure with a structured message that the job can inspect


### 🎯 Challenge 4a: Implement safe table reading with try/except

Write a function `safe_read_table(table_name)` that:
1. Tries to read the table using `spark.table(table_name)`
2. Catches `PySparkException` from `pyspark.errors`
3. Prints a descriptive message with the error class if the table is not found
4. Returns `None` on error, or the DataFrame on success

Test it with a valid table and a non-existent table to confirm both paths work.

**Hints:**
- Import: `from pyspark.errors import PySparkException`
- Use `e.getErrorClass()` to get a structured error code
- Re-raise unexpected errors that you haven't handled

> 🤖 **Databricks Assistant tip:** Ask *"Show me how to catch a PySparkException in Python and print its error class."*

In [ ]:
# YOUR CODE HERE
# Define safe_read_table(table_name) with try/except
# Test with valid and invalid table names


### 🎯 Challenge 4b: Signal task outcome with dbutils.notebook.exit()

When a notebook runs as a Lakeflow Job task, calling `dbutils.notebook.exit()` lets downstream tasks read the outcome using `{{tasks.<task_name>.values.<key>}}`.

Write a function `report_pipeline_outcome(table_name)` that:
1. Reads the specified table and counts its records
2. On success, calls `dbutils.notebook.exit()` with the message `"SUCCESS: Processed <n> records from <table_name>"`
3. On any exception, calls `dbutils.notebook.exit()` with the message `"FAILED: <error message>"`

> ⚠️ **Warning:** `dbutils.notebook.exit()` halts notebook execution immediately. Define the function but **do not call it** — instead, leave the call commented out. In the job, calling it is the correct behaviour.

> 🤖 **Databricks Assistant tip:** Ask *"How does dbutils.notebook.exit work in Azure Databricks when a notebook runs as a job task?"*

In [ ]:
# YOUR CODE HERE
# Define report_pipeline_outcome(table_name)
# Leave the function call commented out


## Exercise 5: Parameterized Notebook Tasks

Reusable pipeline notebooks accept parameters at runtime — making the same notebook work for different scenarios (e.g. different dates, environments, or filter values) without code changes.

Two mechanisms are used in Lakeflow Jobs:
- **`dbutils.widgets`** — pass parameters *into* a notebook from a job task configuration
- **`dbutils.jobs.taskValues`** — pass values *between* notebook tasks in the same job run


### 🎯 Challenge 5a: Add a runtime widget parameter

Add a text widget named `booking_status` with default value `confirmed` and label `Booking Status Filter`.

Then retrieve the widget value and use it to filter `hospitality_lab.silver.bookings_silver`. Display the filtered result and print the record count.

**Hints:**
- `dbutils.widgets.text(name, default, label)`
- `dbutils.widgets.get(name)` returns the current value
- In a Lakeflow Job, the task **Parameters** configuration overrides the default value at runtime

> 🤖 **Databricks Assistant tip:** Ask *"How do I create a text widget in Databricks and retrieve its value to filter a DataFrame?"*

In [ ]:
# YOUR CODE HERE
# Add a widget 'booking_status' (default: 'confirmed', label: 'Booking Status Filter')
# Retrieve the value and filter bookings_silver, then display results


### 🎯 Challenge 5b: Pass values to downstream tasks

After processing, a notebook task can share computed values with downstream tasks using `dbutils.jobs.taskValues.set()`.

Set two task values from the Silver table:
1. Key `silver_record_count` — the number of records in `bookings_silver`
2. Key `properties_processed` — a Python list of distinct `property_id` values

Then print what was set so you can verify it.

> **Note:** `dbutils.jobs.taskValues` is only available inside a Lakeflow Job run. In an interactive notebook session it prints a warning and is a no-op — this is expected.

> 🤖 **Databricks Assistant tip:** Ask *"How do I use dbutils.jobs.taskValues.set to pass a list value from one Databricks notebook task to another?"*

In [ ]:
# YOUR CODE HERE
# Set task values: 'silver_record_count' and 'properties_processed'
# Then print the values to verify
